In [1]:
#. Import existing features.
# No need to regenerate existing features.

'''
Features which need to be regenerated:
1. ViT based bounding box features.
2. Segment anything based segmentation mask features.
3. Zero shot learning based 

Development workflow
1. Import existing codes (and import existing features too)
3. Keep developing the new features one by one into python modules in the folder.
'''

'\nFeatures which need to be regenerated:\n1. ViT based bounding box features.\n2. Segment anything based segmentation mask features.\n3. Zero shot learning based \n\nDevelopment workflow\n1. Import existing codes (and import existing features too)\n3. Keep developing the new features one by one into python modules in the folder.\n'

In [2]:
from PIL import Image, ImageSequence

def load_gif_frames(gif_path):
    """
    Load the frames of a GIF into a list.

    Args:
    - gif_path (str): Path to the GIF file.

    Returns:
    - List[Image.Image]: List of PIL Image objects representing the frames.
    """
    with Image.open(gif_path) as im:
        frames = [frame.copy().convert('RGB') for frame in ImageSequence.Iterator(im)]
    return frames

In [3]:
import torch

def master_feature_generator(current_file, gif_folder, 
                             ml_elements, annotated_data,
                             saving_folder):

    current_dict = torch.load(current_file)

    yt_id = current_dict['metadata']['yt_id']
    frame_index = current_dict['metadata']['frame no.']

    window_size = 5

    # Loading the gif    
    filename = yt_id + '_' + str(frame_index) + '_' + str(window_size) + '.gif'
    import os
    file_location = os.path.join(gif_folder, filename)
    gif_frames = load_gif_frames(file_location)
    
    
    filename_save = yt_id + '_' + str(frame_index) + '_' + str(window_size) + '.pt'
    saving_file = os.path.join(saving_folder , filename_save)
    if os.path.exists(saving_file):
        return 1, 1
    
    

    # 1. Get the features of the bounding boxes in this image.
    from vit import extract_vit_feats
    current_dict = extract_vit_feats(ml_elements, gif_frames, current_dict)
    
    # 2. Get the segmentation masks using segment anything pipeline
    from sam import extract_sam_masks
    current_dict = extract_sam_masks(ml_elements, gif_frames, current_dict)

    # 3. Get the semantic features using zero shot methods
    num_items = len(annotated_data)
    
    for i in range(num_items):
        
        if annotated_data[i]['metadata']['yt_id'] == yt_id:
            if annotated_data[i]['metadata']['frame no.'] == frame_index:
                req_data = annotated_data[i]
    
    hand_indices = []
    object_dict = req_data['bboxes']
    obj_keys = list(object_dict.keys())
    
    for i, o in enumerate(obj_keys):
        temp_cat = object_dict[o]['class']
        if temp_cat == 'hand':
            hand_indices.append(i)
    
    
    # 4. Get semantic embeddings of the bounding boxes
    from semantic import get_semantic_embeddings
    current_dict = get_semantic_embeddings(ml_elements, gif_frames, current_dict, hand_indices)
    
    
    
    ##### Instance Centric Features #####
    # 5. Get the vision embeddings of the joint area.
    from vit_interaction import extract_vit_interaction_feats
    current_dict = extract_vit_interaction_feats(ml_elements, gif_frames, current_dict)
    
    ##### Context-Centric Features #####
    # 6. Activity Embeddings
    activity = current_dict['metadata']['activity name']
    current_dict['activity_embedding'] = ml_elements['Activity_Embedding'][activity]
    

    # 7. Shape Features
    from segmentation_features import get_shape_features_object_centric
    from segmentation_features import get_shape_features_interaction_centric

    frame_width = current_dict['metadata']['frame_width']
    frame_height = current_dict['metadata']['frame_height']
    
    current_dict = get_shape_features_object_centric(current_dict, frame_height,
                                                     frame_width)

    current_dict = get_shape_features_interaction_centric(current_dict)
    # torch.save(current_dict, saving_file)

    return current_dict, 1

In [5]:
# Create all the various ml_elements

ml_elements = {}
device = torch.device('cuda:0')
ml_elements['device'] = device

import clip
clip_model, clip_preproc = clip.load("ViT-B/32")
clip_model = clip_model.eval().to(ml_elements['device'])

ml_elements['VIPLO_CLIP']  = clip_model
ml_elements['VIPLO_PRE_PROC'] = clip_preproc

ml_elements['Activity_Embedding'] = torch.load('/workspace/work/ral_revise_and_resubmit/ral_revise_resubmit/revised_feat_extraction/activity_embeddings.pt')

import sys
sys.path.append("/workspace/work/ral_revise_and_resubmit/segment-anything")

from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "/workspace/work/ral_revise_and_resubmit/segment-anything/sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda:1"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)
ml_elements['SAM_predictor'] = predictor

import torch
import torchvision.models as models
from torchvision.transforms import functional as F


# Initialize the pre-trained classification model
obj_model = models.resnet50(pretrained=True)
obj_model.eval()

# CUDA if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
obj_model = obj_model.to(device)

imagenet_embed_word2vec = torch.load('/workspace/work/ral_revise_and_resubmit/ral_revise_resubmit/revised_feat_extraction/imagenet_word2vec.pt')
imagenet_embed_word2vec = imagenet_embed_word2vec.to(device)


from torchvision.transforms import functional as F
from torchvision import transforms

# Define the transform for preprocessing the RoI
imagenet_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ml_elements['ROI_DET_MODEL'] = obj_model
ml_elements['ROI_PREPROCESS'] = imagenet_preprocess
ml_elements['imagenet_word2vec_embed'] = imagenet_embed_word2vec

hand_embedding = torch.load('/workspace/work/ral_revise_and_resubmit/ral_revise_resubmit/revised_feat_extraction/word2vec_hand_embedding.pt')
hand_embedding = torch.from_numpy(hand_embedding).to(device)

ml_elements['hand_embedding'] = hand_embedding

In [6]:
import warnings

warnings.simplefilter("ignore")

from glob import glob as glob

# old_feat_files = '/workspace/data/data_folder/o2o/ral_features/full_features/*.pt'

old_feat_files = '/workspace/work/server_data_backup/o2o_/gifs_11_features/*.pt'

file_list = glob(old_feat_files)

saving_folder = '/workspace/work/server_data_backup/o2o_/gifs_chi_features'

from tqdm import tqdm as tqdm
gif_folder = '/workspace/work/server_data_backup/o2o_/gifs_11'


fname = '/workspace/work/misc/O2ONet/data/annotation.pkl'
import pickle as pkl

annotated_data = pkl.load( open(fname, 'rb'))

issues = {}
issues['file'] = []
issues['error'] = []

res, success = master_feature_generator(file_list[0], gif_folder, ml_elements, annotated_data, saving_folder)

# for f in tqdm(file_list):

#     try:
#         res, success = master_feature_generator(f, gif_folder, ml_elements,
#                                                 annotated_data, saving_folder)
#         break
#     except Exception as e:
#         print(e)
#         issues['file'].append(f)
#         issues['error'].append(e)
    

# torch.save(issues, 'erroneous_files.pt')

RuntimeError: Trying to create tensor with negative dimension -2: [-2]

In [8]:
r = list(res.keys())

In [10]:
features_list = ['object_i3d_feature', 'bbox_CLIP', 'geometric_feature',
                'object_semantic_embeddings', 'object_centric_shape_feats']

for f in features_list:
    if f not in r:
        print(f)
relative_features_list = ['relative_spatial_feature', 'interaction_bbox_CLIP', 
                        'interaction_centric_shape_feats']

for f in relative_features_list:
    if f not in r:
        print(f)


object_i3d_feature
bbox_CLIP
interaction_bbox_CLIP


In [ ]:
# from glob import glob as glob
# import torch

# old_feat_files = '/workspace/data/data_folder/o2o/ral_features/full_features/*.pt'
# file_list = glob(old_feat_files)

# from tqdm import tqdm as tqdm
# gif_folder = '/workspace/data/data_folder/o2o/gifs_11'

# issues = []

# for f in tqdm(file_list):
#     temp_feat = torch.load(f)
#     im_width, im_height = temp_feat['metadata']['frame_width'], temp_feat['metadata']['frame_height']
#     if im_width!=1280 or im_height!=720:
#         print(im_width, im_height)

In [1]:
pth = '/workspace/work/server_data_backup/o2o_/gifs_11_features/_1_dgZ4-Ldw_1767_5.pt'

import torch

d = torch.load(pth)

In [2]:
d.keys()

dict_keys(['legend', 'metadata', 'num_obj', 'bboxes', 'lr', 'mr', 'cr', 'object_pairs', 'num_relation', 'geometric_feature', 'cnn_bbox_feature', 'iou', 'distance', 'relative_spatial_feature', 'i3d_feature_map', 'motion_feature'])

In [4]:
d['i3d_feature_map'].shape

torch.Size([5, 12, 1024])